In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import glob
import json
import os

import cv2
import numpy as np
from ultralytics import SAM
from ultralytics.utils.ops import ltwh2xyxy

In [ ]:
# ディレクトリの定義

corrected_dir = os.path.join("data", "corrected")
generated_dir = os.path.join("data", "generated")
raw_dir = os.path.join("data", "raw")
segmented_dir = os.path.join("data", "segmented")

In [ ]:
# モデルの読み込み

model_path = os.path.join("weights", "sam2.1_l.pt")
model = SAM(model_path)

In [ ]:
# セグメンテーションの実行

os.makedirs(os.path.join(segmented_dir, "positive"), exist_ok=True)

annotation_path = os.path.join(corrected_dir, "annotations", "train.json")
with open(annotation_path, "r") as f:
    data = json.load(f)

for image in data["images"]:
    id = image["id"]
    file_name = image["file_name"]
    image_path = os.path.join(corrected_dir, "images", file_name)

    annotations = [ann for ann in data["annotations"] if ann["image_id"] == id]

    boxes = [ann["bbox"] for ann in annotations]
    boxes = np.array(boxes)
    boxes = ltwh2xyxy(boxes)

    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    image_bgra = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2BGRA)

    results = model(image_path, bboxes=boxes)

    masks = results[0].masks.data.cpu().numpy()

    for annotation, mask in zip(annotations, masks):
        id = annotation["id"]
        x, y, w, h = [int(b) for b in annotation["bbox"]]
        mask = mask > 0.5
        cropped_image = image_bgra[y : y + h, x : x + w].copy()
        cropped_mask = mask[y : y + h, x : x + w]
        cropped_image[~cropped_mask, 3] = 0

        save_path = os.path.join(segmented_dir, "positive", f"T{id}.png")
        cv2.imwrite(save_path, cropped_image)

In [ ]:
# セグメンテーションの実行

os.makedirs(os.path.join(segmented_dir, "negative"), exist_ok=True)

images = sorted(
    glob.glob(os.path.join(generated_dir, "foregrounds", "*.jpg")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

for i, image in enumerate(images, start=1):
    image_bgr = cv2.imread(image, cv2.IMREAD_COLOR)
    image_bgra = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2BGRA)
    height, width = image_bgra.shape[:2]

    results = model(image, bboxes=[[300, 50, width - 300, height - 50]])

    masks = results[0].masks.data.cpu().numpy()
    index = int(np.argmax(masks.reshape(masks.shape[0], -1).sum(axis=1)))
    mask = masks[index] > 0.5
    
    rows, cols = np.where(mask)

    x = int(cols.min())
    y = int(rows.min())
    w = int(cols.max() - cols.min() + 1)
    h = int(rows.max() - rows.min() + 1)

    cropped_image = image_bgra[y : y + h, x : x + w].copy()
    cropped_mask = mask[y : y + h, x : x + w]
    cropped_image[~cropped_mask, 3] = 0

    save_path = os.path.join(segmented_dir, "negative", f"T{i}.png")
    cv2.imwrite(save_path, cropped_image)